In [ ]:
import os
import time
import toml
import torch
import soundfile as sf
import numpy as np

from models.fastsde import SeldNetSubbandFast

def return_src_posi(index):
    if index in [1, 2, 3, 4, 5]:
        src_posi = np.array([0, 3.0])
    elif index in [6, 7, 8, 9, 10]:
        src_posi = np.array([2.5, 3.0])
    elif index in [11, 12, 13, 14, 15]:
        src_posi = np.array([0.5, 3.0])
    elif index in [16, 17, 18, 19, 20]:
        src_posi = np.array([1, 3.0])
    elif index in [21, 22, 23, 24, 25]:
        src_posi = np.array([1.5, 3.0])
    elif index in [26, 27, 28, 29, 30]:
        src_posi = np.array([2.0, 3.0])
    elif index == 31:
        src_posi = np.array([2.0, 2.5])
    elif index == 32:
        src_posi = np.array([2.5, 2.5])
    elif index == 33:
        src_posi = np.array([1.5, 2.5])
    elif index == 34:
        src_posi = np.array([2.0, 2.0])
    elif index == 35:
        src_posi = np.array([2.5, 2.0])
    elif index == 36:
        src_posi = np.array([2.5, 1.5])
    elif index == 37:
        src_posi = np.array([2.5, 1.5])
    return src_posi

def read_txt_lines(txt_path):
    with open(txt_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    lines = [line.rstrip('\n') for line in lines]
    out = np.array([list(map(float, s.split())) for s in lines])
    return out

def recursive_average(nums):
    if len(nums) < 2:
        return nums
    new_nums = [(nums[i] + nums[i + 1]) / 2 for i in range(len(nums) - 1)]
    return recursive_average(new_nums)

os.environ['PATH'] = '/sbin:' + os.environ.get('PATH', '')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# configs
config = toml.load('./configs/real_audio/config.toml')

model = SeldNetSubbandFast(
    features_set="all",
    n_subbands=8,   
    c_mid=24,
    n_blocks=3,
    fuse_c=64,
    use_gru=True,   
    gru_hidden=96,
    att_conf="Nothing",
).to(device)

# # UltraFast
# model = SeldNetSubbandFast(
# features_set="all",
# n_subbands=6,   
# c_mid=16,
# n_blocks=2,
# fuse_c=48,
# use_gru=False,  
# att_conf="Nothing",).to(device)


checkpoint_path ='PATH/model_0400_loss_0.0074.tar'
checkpoint = torch.load(checkpoint_path, map_location=device)
model_weights = checkpoint['model']
model.load_state_dict(model_weights, strict=True)
model.eval()
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('total_params',total_params)

# test dataset
audio_root_dirs   = './real_record/audio'
mic_pos_root_dirs = './real_record/mic_pos'
test_list         = [26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36] 
distant_est_result= {}
for i in test_list:
    print(f"real world set {i}:")
    used_file_index = [i] 
    interval = 0.1                # The recording interval is taken once every 100ms
    used_internal = 0.6           # An interval of 400ms used
    samplerate = 16000

    # total used audio & mic_pos
    total_used_audio = []
    total_posi = []

    for index in used_file_index:
        audio_path = os.path.join(audio_root_dirs, f"recorded_audio{index}.wav")
        posi_path = os.path.join(mic_pos_root_dirs, f"mic_pos_{index}.txt")
        src_posi = return_src_posi(index)

        audio, samplerate = sf.read(audio_path) # [wave length, channel]
        posis = read_txt_lines(posi_path)       # [len posi, 2] ([x, y])

        audio_slicses = []
        audio_step = int(used_internal * samplerate)
        audio_hop = audio_step
        # audio_hop = int(0.1 * samplerate)                            
        audio_used_num = int((len(audio)) // audio_hop)
        for i in range(0, audio_used_num):
            audio_slicses.append(audio[i * audio_hop : i * audio_hop + audio_step])


        posi_slicses = []
        posi_step = int(used_internal / interval)
        posi_hop = posi_step #1
        posi_used_num = int((len(posis)) // posi_hop)

        for i in range(0, posi_used_num):
            used_x_posis = posis[i * posi_hop : i * posi_hop + posi_step, 0] #　list
            used_y_posis = posis[i * posi_hop : i * posi_hop + posi_step, 1] #　list
            
            tgt_x_dis = recursive_average(used_x_posis)
            tgt_y_dis = recursive_average(used_y_posis)
            posi_slicses.append(np.array([tgt_x_dis, tgt_y_dis]))


        total_used_audio.append(audio_slicses)
        total_posi.append(posi_slicses)

    total_esti_dis = []
    total_real_dis = []
    total_error = []
    start_time = time.time()
    for index in range(len(audio_slicses)):
        audio_input = torch.tensor(audio_slicses[index], dtype=torch.float32).unsqueeze(0).to(device) # [1, wave]
        esti_dis, all_value = model(audio_input, info = 1)

        # esti & real
        esti_dis = esti_dis.squeeze(0).cpu().detach().numpy() # 
        total_esti_dis.append(esti_dis)
        real_posi = posi_slicses[index]
        real_dis = np.sqrt((src_posi[0] - real_posi[0])**2 + (src_posi[1] - real_posi[1])**2)
        total_real_dis.append(real_dis)

        error = np.abs(esti_dis - real_dis)
        total_error.append(error)

        if index % 500 == 0 and index > 0 :
            end_time = time.time()
            # print(f'index: {index}, esti_dis: {esti_dis}m, real_dis: {real_dis}m, error: {error}m, ave_time: {(end_time-start_time)/500}s')
            start_time = time.time()

    mean_error = sum(total_error) / len(total_error)
    print('mean_error', mean_error)
    data = {'posi_slicses': posi_slicses,
        'total_esti_dis': total_esti_dis,
        'total_real_dis': total_real_dis,
        'total_error': total_error,
        'src_posi': src_posi}
    distant_est_result[str(used_file_index[0])] = data

In [ ]:
import matplotlib.pyplot as plt
for j in distant_est_result.keys():
    print(f"------------dataset {j}-------------------")
    data_error  = np.zeros((1,0))
    loaded_data = distant_est_result[j]

    total_esti_dis = loaded_data['total_esti_dis']
    posi_slicses   = loaded_data['posi_slicses'][:len(total_esti_dis)]
    total_real_dis = loaded_data['total_real_dis']
    total_error    = loaded_data['total_error']
    src_posi       = loaded_data['src_posi']

    total_esti_dis = np.array(loaded_data['total_esti_dis'])
    total_real_dis = np.array(loaded_data['total_real_dis'])
    total_error = np.array(loaded_data['total_error'])

    plt.figure(figsize=(6, 4))
    # Draw the estimated distance and the actual distance
    plt.plot(total_esti_dis, label='Estimated Distance', color='blue', alpha = 0.5)
    plt.plot(total_real_dis, label='Real Distance', color='green')
    plt.plot(total_error, label='Error', color='red', linestyle='--')

    plt.xlabel('Index')
    plt.ylabel('Distance')
    plt.title('Distance Comparison and Error')
    plt.legend()
    plt.grid(True)
    plt.show()

    total_posi_slicses = np.array(loaded_data['posi_slicses'][:len(total_esti_dis)])
    src_posi = np.array(loaded_data['src_posi'])


    # plot trajectory and estimated position
    total_posi_slicses = np.array(loaded_data['posi_slicses'])
    plt.figure(figsize=(4, 4))
    plt.scatter(src_posi[0], src_posi[1], c='red', marker='o', label='Source')
    plt.scatter(total_posi_slicses[:, 0], total_posi_slicses[:, 1], c='blue', marker='o', label='Position Slices')

    plt.xlabel('X')
    plt.ylabel('Y')
    plt.title('Position Slices Scatter Plot')
    plt.grid(True)
    plt.axis('equal')
    plt.legend()
    plt.show()
